In [ ]:
# Instalación de librerías necesarias (si estás en Colab o no las tienes)
#!pip install nltk scikit-learn pandas matplotlib seaborn wordcloud arxiv

import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Librerías de NLP
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

# Librerías de Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Descargar recursos de NLTK necesarios
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

print("✅ Librerías importadas y recursos descargados.")

✅ Librerías importadas y recursos descargados.


[nltk_data] Error loading omw-ont: Package 'omw-ont' not found in
[nltk_data]     index
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/debian12/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [11]:
import nltk

# Para la tokenización (causante de tu error)
print("⏳ Descargando recurso 'punkt_tab'...")
nltk.download('punkt_tab', quiet=True) 

# Para la lematización (probablemente necesario para tu función)
print("⏳ Descargando recurso 'wordnet'...")
nltk.download('wordnet', quiet=True)

print("\n✅ Recursos NLTK esenciales descargados.")

⏳ Descargando recurso 'punkt_tab'...
⏳ Descargando recurso 'wordnet'...

✅ Recursos NLTK esenciales descargados.


In [12]:
# Cargar el dataset
try:
    df = pd.read_csv("arxiv_Radio_Astronomy.csv")
    # Eliminar filas vacías si las hay
    df = df.dropna(subset=['abstract'])
    print(f" Datos cargados: {len(df)} abstracts listos para procesar.")
except FileNotFoundError:
    print(" No se encontró el archivo 'arxiv_Radio_Astronomy.csv'.")
    print("Por favor, asegúrate de haber ejecutado la recolección de datos primero.")

 Datos cargados: 1039 abstracts listos para procesar.


In [13]:
# 1. Configuración de Stopwords (palabras vacías)
stop_words = set(stopwords.words('english'))
# Palabras académicas que no aportan significado al tema
academic_words = {
    'et', 'al', 'fig', 'figure', 'table', 'section', 'paper', 'present',
    'results', 'show', 'study', 'method', 'using', 'used', 'based',
    'data', 'new', 'approach', 'proposed', 'also', 'however', 'analysis',
    'work', 'demonstrate', 'observed', 'observation'
}
stop_words.update(academic_words)

# 2. Función auxiliar para mapear etiquetas de NLTK a WordNet
def get_wordnet_pos(word):
    """Mapea el tag POS de NLTK al formato que entiende el Lematizador"""
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

# 3. Función principal de limpieza
def clean_text_advanced(text):
    if not isinstance(text, str): return ""
    
    # A. Limpieza básica
    text = text.lower()
    text = re.sub(r'http\S+', '', text) # Eliminar URLs
    text = re.sub(r'arxiv:\S+', '', text) # Eliminar IDs arxiv
    text = re.sub(r'\$.*?\$', '', text) # Eliminar fórmulas LaTeX
    text = re.sub(r'\[.*?\]', '', text) # Eliminar referencias [1]
    
    # B. Unificar términos clave con guiones
    # Esto ayuda a que "radio-telescope" y "radiotelescope" sean lo mismo
    text = text.replace('-', ' ') 
    
    # C. Eliminar caracteres no alfanuméricos
    text = re.sub(r'[^a-z\s]', '', text)
    
    # D. Tokenización y Lematización inteligente
    tokens = word_tokenize(text)
    lemmatizer = WordNetLemmatizer()
    clean_tokens = []
    
    for token in tokens:
        if token not in stop_words and len(token) > 2:
            # Lematizar según su rol gramatical (su verbo, sustantivo, etc.)
            lemma = lemmatizer.lemmatize(token, get_wordnet_pos(token))
            clean_tokens.append(lemma)
            
    return ' '.join(clean_tokens)

# Aplicar al DataFrame
print("⏳ Procesando textos (puede tardar unos segundos)...")
df['processed_abstract'] = df['abstract'].apply(clean_text_advanced)
print("✅ Limpieza completada.")

# Verificación
print(f"\nOriginal: {df['abstract'].iloc[0][:150]}...")
print(f"Procesado: {df['processed_abstract'].iloc[0][:150]}...")

⏳ Procesando textos (puede tardar unos segundos)...


LookupError: 
**********************************************************************
  Resource [93maveraged_perceptron_tagger_eng[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('averaged_perceptron_tagger_eng')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtaggers/averaged_perceptron_tagger_eng[0m

  Searched in:
    - '/home/debian12/nltk_data'
    - '/home/debian12/Documentos/UD/Mineria_de_Datos/MD/nltk_data'
    - '/home/debian12/Documentos/UD/Mineria_de_Datos/MD/share/nltk_data'
    - '/home/debian12/Documentos/UD/Mineria_de_Datos/MD/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************
